# Setup

In [1]:
from pprint import pprint
from dotenv import dotenv_values


cfg = dotenv_values()

SEED = cfg['SEED']
CHAT_SMALL = cfg['CHAT_SMALL']
CHAT_MODEL = cfg['CHAT_MODEL']

# 1. Simple Agent

- Chat model: for conversation
- Tools: a simple tool for searching wikipedia
- Memory: a simple in-memory state, conversation history with summarization

In [2]:
from langchain_ollama import ChatOllama
from langchain.messages import SystemMessage, HumanMessage
from langchain_core.messages.utils import count_tokens_approximately
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import (
    ToolCallLimitMiddleware,
    ToolErrorMiddleware,
    ToolCallRequest,
    SummarizationMiddleware,
)
from langgraph.checkpoint.memory import InMemorySaver
import wikipediaapi


# 1. Chat model
chat = ChatOllama(
    model=CHAT_MODEL,
    reasoning=False,
    seed=SEED,
)

# 2. Tools
wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainPrac/1.0',
    language='en',
)

@tool
def search_wiki(keyword: str) -> str:
    """Search wiki by keyword and return text."""
    pages = wiki.search(keyword, limit=2).pages

    results = []
    for keyword, value in pages.items():
        results.append(f"{value.text[:200]} ...\n")

    return results

# 3. Agent
system_prompt = SystemMessage(
    content=(
        "You MUST use provided tool to search the wiki "
        "before answer user question."
    )
)

def on_error(exc: Exception, request: ToolCallRequest) -> str | None:
    return (
        "Error for searching:\n"
        f"Tool: {request.tool_call['name']}, "
        f"Error: {type(exc).__name__}"
    )

agent = create_agent(
    model=chat,
    tools=[search_wiki],
    system_prompt=system_prompt,
    middleware=[
        ToolCallLimitMiddleware(run_limit=3),
        ToolErrorMiddleware(on_error=on_error),
        SummarizationMiddleware(
            model=ChatOllama(
                model=CHAT_MODEL,
                reasoning=False,
            ),
            trigger=('tokens', 200),
            keep=('tokens', 100),
            token_counter=count_tokens_approximately,
            trim_tokens_to_summarize=1000,
            summary_prompt=(
                "Summarize conversation history up to 150 tokens. "
                "Preserve the user's questions and shortly describe the situation. "
                "Do not invent anything not in the user prompt.\n\n"
                "{messages}"
            ),
        ),
    ],
    checkpointer=InMemorySaver(),
)

# 4. Invoke
config = {"configurable": {"thread_id": "user-1"}}

resp = agent.invoke(
    input={
        'messages': [HumanMessage("What is retrieval-augmented generation?")],
    },
    config=config,
)

for msg in resp['messages']:
    msg.pretty_print()

print("-"*80, "\n\n\n")

resp = agent.invoke(
    input={
        'messages': [HumanMessage("Which keyword did the user ask before?")],
    },
    config=config,
)

for msg in resp['messages']:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

User asks: "What is retrieval-augmented generation?"  
Situation: The user seeks a definition or explanation of retrieval-augmented generation (RAG), a technique that combines retrieval of relevant external information with generative models to produce responses. No additional context or follow-up questions are provided in the prompt.
================================== Ai Message ==================================
Tool Calls:
  search_wiki (8cf7612b-8b09-4fb6-b233-f80902bf5b78)
 Call ID: 8cf7612b-8b09-4fb6-b233-f80902bf5b78
  Args:
    keyword: retrieval-augmented generation
================================= Tool Message =================================
Name: search_wiki

["Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first

# 2. RAG

In [3]:
import wikipediaapi
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import models
from sentence_transformers import CrossEncoder
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore


# 1. Documents
wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainRAG/1.0',
    language='en',
)
pages = wiki.search('Retrieval-augmented generation', limit=3).pages
docs = [
    Document(
        page_content=page.text,
        metadata={'title': title},
    )
    for title, page in pages.items()
]

# 2. Split & Embedding
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=64,
)
chunks = splitter.split_documents(docs)

emb_dense = OllamaEmbeddings(model='qwen3-embedding:0.6b')
emb_sparse = FastEmbedSparse('Qdrant/bm25')

# 3. Store & Vector DB
vector_db = QdrantVectorStore.from_documents(   # .from_existing_collection()
    collection_name='wiki_hybrid',
    # location=':memory:',
    path='../../tmp/rag_prac',
    documents=chunks,
    embedding=emb_dense,
    sparse_embedding=emb_sparse,
    sparse_vector_params={'modifier': models.Modifier.IDF},
    retrieval_mode=RetrievalMode.HYBRID,
)

# 4. Retriever & Reranker
retriever = vector_db.as_retriever(search_kwargs={"k": 5})
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L6-v2')

@tool
def search_context(query: str) -> str | None:
    """Use user query to search related documents."""
    retrieved_docs = retriever.invoke(query)
    if not retrieved_docs:
        return "No documents found."

    ranked = reranker.rank(
        query="What is RAG?",
        documents=[doc.page_content for doc in retrieved_docs],
        top_k=2,
    )
    reranked_docs = [retrieved_docs[item['corpus_id']] for item in ranked]
    context = "\n\n".join(
        f"Title: {doc.metadata['title']}\n{doc.page_content}"
        for doc in reranked_docs
    )

    return context

# 5. Agent
system_prompt = SystemMessage(
    content=(
        "You must use provided tool to search the information, "
        "and answer the user question shortly."
    )
)

agent = create_agent(
    model=ChatOllama(
        model=CHAT_MODEL,
        reasoning=False,
    ),
    tools=[search_context],
    system_prompt=system_prompt,
)

# 6. Invoke
resp = agent.invoke(
    input={
        'messages': [HumanMessage("Search and answer. What is RAG?")],
    }
)

for msg in resp['messages']:
    msg.pretty_print()

W1112 15:14:54.719000 29408 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:14:54.831000 29408 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:14:56.303000 29408 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

================================ Human Message =================================

Search and answer. What is RAG?
================================== Ai Message ==================================
Tool Calls:
  search_context (85d984ab-0c0a-45be-866e-70cc740601c5)
 Call ID: 85d984ab-0c0a-45be-866e-70cc740601c5
  Args:
    query: What is RAG?
================================= Tool Message =================================
Name: search_context

Title: Retrieval-augmented generation
Retrieval-augmented generation (RAG) is a technique that enables large language models (LLMs) to retrieve and incorporate new information from external data sources. With RAG, LLMs first refer to a specified set of documents, then respond to user queries. These documents supplement information from the LLM's pre-existing training data. This allows LLMs to use domain-specific and/or updated information that is not available in the training data. For example, this enables LLM-based chatbots to access internal

Tit

# 3. Memory

In [4]:
from pydantic import BaseModel
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.messages import SystemMessage, HumanMessage
from langchain.agents import create_agent
from langmem import (
    create_memory_store_manager,
    create_search_memory_tool,
)
import sqlite3
from langgraph.store.sqlite import SqliteStore


# 1. Memory schema
class Fact(BaseModel):
    """One specific fact."""
    content: str

# 2. Store
conn = sqlite3.connect(
    "../../tmp/memory_prac.sqlite",
    check_same_thread=False,
    isolation_level=None,
)
store = SqliteStore(
    conn=conn,
    index={
        'embed': OllamaEmbeddings(model='qwen3-embedding:0.6b'),
        'dims': 1024,
    }
)
store.setup()

# 3. Memory manager
namespace = ('memory_prac', 'user-1', 'fact')

memory_manager = create_memory_store_manager(
    ChatOllama(model=CHAT_MODEL, reasoning=False),
    schemas=[Fact],
    instructions=(
        "Call the Fact tool for each explicit fact including their name."
    ),
    enable_inserts=True,
    enable_deletes=False,
    query_limit=3,
    store=store,
    namespace=namespace,
)

# 4. Agent
search_tool = create_search_memory_tool(
    store=store,
    namespace=namespace,
    instructions="Use this tool to retrieve user memory.",
)

agent = create_agent(
    model=ChatOllama(model=CHAT_MODEL, reasoning=False),
    tools=[search_tool],
    system_prompt=SystemMessage(
        content=(
            "You must use provided tool to retrieve memory before answer user question."
        )
    ),
)

# 5. Invoke
history = "My name is yana."
memory_manager.invoke(
    input={'messages': [HumanMessage(history)]}
)

for memory in store.search(namespace, query=history, limit=3):
    print(memory.value)

resp = agent.invoke(
    input={'messages': [HumanMessage("Use provided tool to search my name.")]}
)

for msg in resp['messages']:
    msg.pretty_print()

{'kind': 'Fact', 'content': {'content': 'My name is yana.'}}
================================ Human Message =================================

Use provided tool to search my name.
================================== Ai Message ==================================
Tool Calls:
  search_memory (fe7905e5-42f6-4264-b554-a00b55c7e94e)
 Call ID: fe7905e5-42f6-4264-b554-a00b55c7e94e
  Args:
    query: my name
================================= Tool Message =================================
Name: search_memory

[{"namespace":["memory_prac","user-1","fact"],"key":"bdaeb156-0ce0-452d-be0d-d13616e4af4b","value":{"kind":"Fact","content":{"content":"My name is yana."}},"created_at":"2026-11-12T06:15:20","updated_at":"2026-11-12T06:15:20","score":0.7533086985349655}]
================================== Ai Message ==================================

My name is Yana.


# 4. GraphRAG

In [2]:
# 1. Documents
import wikipediaapi
from langchain_core.documents import Document

wiki = wikipediaapi.Wikipedia(
    user_agent='LangChainRAG/1.0',
    language='en',
)
pages = wiki.search('Retrieval-augmented generation', limit=3).pages
docs = [
    Document(
        page_content=page.text,
        metadata={'title': title},
    )
    for title, page in pages.items()
]

In [15]:
# 2. Build the graph
import asyncio
from llama_index.core import Document as LlamaDocument, PropertyGraphIndex
from llama_index.core.indices.property_graph import SimpleLLMPathExtractor
from llama_index.llms.ollama import Ollama

llm = Ollama(model=CHAT_MODEL, temperature=0, context_window=4096, request_timeout=180)
extractor = SimpleLLMPathExtractor(
    llm=llm, max_paths_per_chunk=5,
    extract_prompt=(
        'Extract up to {max_knowledge_triplets} facts explicitly stated in this passage. '
        'Write one (subject, predicate, object) per line. If there are no facts, return nothing.\n'
        'Passage:\n{text}'
    ),
)
index = await asyncio.to_thread(
    PropertyGraphIndex.from_documents,
    [LlamaDocument(text=doc.page_content) for doc in docs],
    llm=llm, kg_extractors=[extractor], embed_kg_nodes=False,
)

In [16]:
# 3. Retrieve graph facts
query = 'How does retrieval-augmented generation (RAG) use external information to answer a question?'
retriever = index.as_retriever(include_text=False, use_async=False, limit=3)
facts = retriever.retrieve(query)

store = index.property_graph_store
all_triples = store.graph.get_triplets()
fact_texts = {fact.text for fact in facts}
retrieved_triples = [triple for triple in all_triples
                     if ' -> '.join(part.id for part in triple) in fact_texts]

print('Query:', query)
print(f'Retrieved facts ({len(facts)})')
for fact in facts:
    print('-', fact.text)

Query: How does retrieval-augmented generation (RAG) use external information to answer a question?
Retrieved facts (3)
- Retrieval-augmented generation -> Allows -> Llms to include sources in their responses for user verification.
- Graphrag -> Extends -> Rag
- Retrieval-augmented generation -> Enables -> Large language models to retrieve and incorporate new information from external data sources.


In [17]:
# 4. Generate an answer from the facts and their source passages
from llama_index.core import PromptTemplate, get_response_synthesizer
from llama_index.core.graph_stores.types import TRIPLET_SOURCE_KEY
from llama_index.core.schema import NodeWithScore

source_ids = {rel.properties[TRIPLET_SOURCE_KEY] for _, rel, _ in retrieved_triples}
passages = store.get_llama_nodes(list(source_ids)) if source_ids else []
evidence = facts + [NodeWithScore(node=passage) for passage in passages]

answer_prompt = PromptTemplate(
    'Answer using only the evidence below. Explain how the steps connect in 3 to 5 sentences when supported. '
    'Include a concrete detail and explain why it matters when the passages support it. '
    'If the evidence is incomplete, say what is missing.\n\n'
    'Evidence:\n{context_str}\n\nQuestion: {query_str}\nAnswer:'
)
synthesizer = get_response_synthesizer(llm=llm, text_qa_template=answer_prompt)
response = synthesizer.synthesize(query, nodes=evidence)
print(f'Answer (facts: {len(facts)}, source passages: {len(passages)}):\n{response}')

Answer (facts: 3, source passages: 2):
Retrieval-augmented generation (RAG) uses external information by first retrieving relevant documents—such as databases or uploaded files—through a document retriever that matches a user’s query to stored text embeddings. These retrieved documents are then fed into the LLM via prompt engineering, allowing the model to generate responses based on real-world, up-to-date sources rather than relying solely on its static training data. A concrete detail is that RAG can pull information from internal company documents or authoritative sources, which matters because it enables accurate, context-specific answers—like a chatbot citing a company policy that was recently updated—without needing to retrain the entire model. This process reduces AI hallucinations and allows users to verify the sources, increasing trust and accuracy. The connection between steps is clear: retrieval provides real-time data, which is then used to refine the prompt, leading to a m

In [26]:
# 5. Visualize the full and retrieved graphs
from pathlib import Path
import networkx as nx
from pyvis.network import Network

G = nx.MultiDiGraph()
for source, relation, target in all_triples:
    G.add_edge(source.id, target.id, key=relation.id, label=relation.label)
for attrs in G.nodes.values():
    attrs.update(shape='box', widthConstraint={'maximum': 180})

retrieved = G.edge_subgraph((s.id, t.id, r.id) for s, r, t in retrieved_triples)

for name, graph in [('full', G), ('retrieved', retrieved)]:
    view = Network(height='95vh', directed=True, cdn_resources='in_line')
    view.from_nx(graph.copy())
    view.barnes_hut(gravity=-6000, spring_length=220, overlap=1)
    Path(f'graph_{name}.html').write_text(view.generate_html(), encoding='utf-8')